[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C32_Skills_Tools_Course/02_slash_commands/02_slash_commands.ipynb)

# 02 · Slash 命令与动态注入

目标：用**纯标准库**（`shlex`/`re`/`subprocess`）从零写出 slash 命令系统的完整链路——**解析 → 参数绑定 → 动态注入(!`shell` / @file) → 路由 → 展开成 prompt**，全程用 **MockLLM** 当模型、`assert` 验证，**无需 API key、不依赖任何框架**。

路线：解析器 → 参数绑定器 → 动态注入器（可插拔执行器，安全可测）→ 命令路由器 → 接 MockLLM 端到端 → ✏️ 练习 → 📖 答案 → 🧪 真实命令文件胶囊。

> 心智模型：**slash 命令 = 把用户常重复的工作流固化成「带占位符 + 动态注入的 prompt 模板」，敲一下就展开成完整指令**。难点在「调用模型」之前的文本解析、绑定、注入、分发。

> 对照模块 01：**skill 由模型按相关性自动触发、不带参数；slash 命令由用户显式调用、带参数且能动态注入**。两条入口互补。

## 1 · 解析：把一行命令拆成名字与参数

用户输入 `/review src/app.py "fix login bug"`。解析要：判断是不是 slash 命令、取命令名、把剩下的拆成参数。

**难点**：带引号的参数要当作一个整体。手写空白切分会拆错——用标准库 `shlex`（懂引号与转义）。残缺引号要报错而非崩溃。

In [ ]:
import json, re, shlex, subprocess, os

class MockLLM:
    '''确定性假模型: 按规则把 prompt 映射到响应。形状贴近真实 tool-use。'''
    def __init__(self, rules, default=None):
        self.rules = rules
        self.default = default or {'type': 'text', 'text': '(no rule matched)'}
        self.calls = 0
    def __call__(self, prompt):
        self.calls += 1
        text = prompt if isinstance(prompt, str) else json.dumps(prompt, ensure_ascii=False)
        for kw, resp in self.rules:
            if kw in text:
                return json.loads(json.dumps(resp))
        return json.loads(json.dumps(self.default))

def parse_command(line):
    '''解析一行 slash 命令。返回 {'name','args'} 或 None(非命令)；残缺引号抛 ValueError。'''
    line = line.strip()
    if not line.startswith('/'):
        return None                       # 不是 slash 命令
    body = line[1:]
    parts = shlex.split(body)             # 懂引号: 把带空格的引号串当作一个整体
    if not parts:
        return None                       # 纯 '/' 或空
    return {'name': parts[0], 'args': parts[1:]}

print(parse_command('/review src/app.py "fix login bug"'))
print(parse_command('普通聊天, 不是命令'))
print(parse_command('/'))
r = parse_command('/review src/app.py "fix login bug"')
assert r == {'name': 'review', 'args': ['src/app.py', 'fix login bug']}   # 引号串保整体!
assert parse_command('普通聊天') is None
assert parse_command('/') is None and parse_command('   ') is None
# 残缺引号: shlex 抛 ValueError, 我们让上层捕获(见路由)
raised = False
try:
    parse_command('/x "unclosed')
except ValueError:
    raised = True
assert raised, '残缺引号应抛 ValueError'
print('✅ 解析器: 取名字+拆参数, 引号串保整体, 非命令/空返回 None, 残缺引号报错')

## 2 · 绑定：把参数填进占位符

三类占位符（Claude Code 约定）：`$ARGUMENTS`=全部参数、`$1`/`$2`=位置参数、越界的 `$N` 填空串。

**顺序**：先替 `$ARGUMENTS`（最长标记），再用正则一次性替所有 `$<数字>`（正确处理 `$10` 这类多位数）。

In [ ]:
def bind_args(template, args):
    '''把 args 绑定进模板的 $ARGUMENTS / $1 / $2 ... 占位符。越界位置参数填空串。'''
    out = template.replace('$ARGUMENTS', ' '.join(args))
    def repl(m):
        i = int(m.group(1))               # $1 -> args[0]
        return args[i - 1] if 1 <= i <= len(args) else ''
    return re.sub(r'\$(\d+)', repl, out)  # 整体匹配数字: $10 -> 第10个, 不是 $1+'0'

tmpl = '评审 $1 (备注: $2)；全部参数=[$ARGUMENTS]'
print(bind_args(tmpl, ['app.py', 'urgent']))
print(bind_args(tmpl, ['app.py']))            # $2 越界 -> 空
assert bind_args(tmpl, ['app.py', 'urgent']) == '评审 app.py (备注: urgent)；全部参数=[app.py urgent]'
assert bind_args(tmpl, ['app.py']) == '评审 app.py (备注: )；全部参数=[app.py]'   # $2 填空串
# 多位数占位符: 给 11 个参数, $11 应取第 11 个
many = [f'a{i}' for i in range(1, 12)]
assert bind_args('第十一个=$11', many) == '第十一个=a11'
assert bind_args('空模板没有占位符', ['x']) == '空模板没有占位符'
print('✅ 绑定器: $ARGUMENTS / $1 / 越界填空 / 多位数 $11 全部正确')

## 3 · 动态注入：把实时上下文嵌进 prompt（安全可测）

两类动态标记：`` !`<shell>` `` 注入命令的 stdout、`@<path>` 注入文件内容。

**关键设计**：把『怎么执行 / 怎么读』做成**可注入的回调**（依赖注入），而不是写死 `subprocess`。测试时传**确定性假执行器**——既安全（绝不真跑危险命令）又可测。

In [ ]:
def inject_dynamic(text, runner, reader):
    '''展开 text 里的 !`cmd` 与 @path。
       runner(cmd) -> str(命令输出); reader(path) -> str(文件内容)。
       注入逻辑与『怎么执行/怎么读』解耦, 故可传安全的桩来测试。'''
    # 1) !`...` -> runner 的输出
    text = re.sub(r'!`([^`]+)`', lambda m: runner(m.group(1)), text)
    # 2) @path -> reader 的内容 (路径取非空白串)
    text = re.sub(r'@(\S+)', lambda m: reader(m.group(1)), text)
    return text

# 确定性假执行器/读取器(安全桩): 绝不真跑命令/读盘, 输出可预测、可断言
def fake_runner(cmd):
    table = {'git diff app.py': '- old\n+ new', 'date': '2024-01-01'}
    return table.get(cmd, f'<output of: {cmd}>')
def fake_reader(path):
    table = {'docs/checklist.md': '1. 有测试吗\n2. 命名清晰吗'}
    return table.get(path, f'<content of: {path}>')

t = '请评审 app.py。diff:\n!`git diff app.py`\n清单:\n@docs/checklist.md'
expanded = inject_dynamic(t, fake_runner, fake_reader)
print(expanded)
assert '- old\n+ new' in expanded          # !`git diff` 的输出被注入
assert '1. 有测试吗' in expanded             # @checklist 的内容被注入
assert '!`' not in expanded and '@docs' not in expanded   # 标记已被替换掉
print('✅ 动态注入: !`shell` 与 @file 被求值替换; 执行器可插拔(安全可测)')

## 4 · 一个真实但确定的执行器（subprocess，离线可跑）

第 3 节用假执行器测逻辑。真实执行器用 `subprocess`，但**绝不跑任意/危险命令**——本课用一个**白名单**只放行安全、确定、离线可得的命令（如 `python3 -c "print(...)"`），保证 0 失败且无副作用。

> ⚠️ 生产里 `!`shell`` 是任意命令执行，必须白名单/沙箱/确认——这里正是示范这种受控执行。

In [ ]:
def safe_subprocess_runner(cmd, allow_prefixes=('python3 -c', 'echo')):
    '''只放行白名单前缀的命令, 用 subprocess 跑, 返回 stdout。其它一律拒绝(不执行)。'''
    if not any(cmd.startswith(p) for p in allow_prefixes):
        return f'[拒绝执行: 不在白名单] {cmd}'      # 危险命令挡在执行之外
    try:
        out = subprocess.run(shlex.split(cmd), capture_output=True, text=True, timeout=5)
        return out.stdout.strip()
    except Exception as e:
        return f'[执行错误] {type(e).__name__}: {e}'

# 白名单内: python3 -c 打印, 确定性、离线可得
out_ok = safe_subprocess_runner('python3 -c "print(2+3)"')
print('白名单命令输出:', repr(out_ok))
assert out_ok == '5'
# 白名单外: 危险命令被拒绝, 绝不执行
out_no = safe_subprocess_runner('rm -rf /tmp/whatever')
print('危险命令:', out_no)
assert out_no.startswith('[拒绝执行')
# 用真实执行器做一次动态注入
t2 = '结果是 !`python3 -c "print(6*7)"`'
exp2 = inject_dynamic(t2, safe_subprocess_runner, fake_reader)
assert exp2 == '结果是 42'
print('✅ 受控执行器: 白名单放行确定命令、拒绝危险命令; 动态注入端到端跑通')

## 5 · 路由：把命令分发到处理器

命令注册表：`命令名 → {模板, 元数据}`。路由器解析→查表→绑定→注入，**任何错误都返回结构化结果（带 ok 标志）而非抛异常**——让上层能优雅提示『未知命令』。

In [ ]:
class CommandRouter:
    def __init__(self):
        self.commands = {}                          # name -> {template, meta}
    def register(self, name, template, meta=None):
        self.commands[name] = {'template': template, 'meta': meta or {}}
    def names(self):
        return sorted(self.commands)
    def dispatch(self, line, runner, reader):
        '''解析→查表→绑定→注入。返回 {'ok':bool, 'prompt'|'error':...}。绝不向上抛。'''
        try:
            parsed = parse_command(line)
        except ValueError as e:
            return {'ok': False, 'error': f'解析失败: {e}'}
        if parsed is None:
            return {'ok': False, 'error': '不是 slash 命令'}
        name = parsed['name']
        if name not in self.commands:
            return {'ok': False, 'error': f'未知命令: /{name}'}   # 挡住打错的命令
        bound = bind_args(self.commands[name]['template'], parsed['args'])
        prompt = inject_dynamic(bound, runner, reader)
        return {'ok': True, 'prompt': prompt}

router = CommandRouter()
router.register('review', '评审 $1。diff:\n!`git diff $1`', {'description': '代码评审'})
router.register('greet', '向 $1 问好', {'description': '打招呼'})

ok = router.dispatch('/greet Alice', fake_runner, fake_reader)
print(ok)
bad1 = router.dispatch('/unknown x', fake_runner, fake_reader)
bad2 = router.dispatch('随便说点啥', fake_runner, fake_reader)
print(bad1); print(bad2)
assert ok['ok'] is True and ok['prompt'] == '向 Alice 问好'
assert bad1['ok'] is False and '未知命令' in bad1['error']
assert bad2['ok'] is False
assert router.names() == ['greet', 'review']
print('✅ 路由器: 注册/分发/绑定/注入; 未知命令与非命令安全返回 ok=False, 绝不崩溃')

## 6 · 接进 agent：展开后的 prompt 交给（Mock）模型

slash 命令的产物是**一段展开后的 prompt**——与任何普通用户输入无异。把它喂给 MockLLM，模型据此回应或调用工具。这就是命令系统接进 agent 的全部。

In [ ]:
def run_slash(line, router, llm, runner, reader):
    '''完整一轮: 命令 -> 展开 prompt -> 喂给模型 -> 返回模型响应。'''
    res = router.dispatch(line, runner, reader)
    if not res['ok']:
        return {'type': 'text', 'text': '[命令错误] ' + res['error']}
    return llm(res['prompt'])            # 展开后的 prompt 就是普通输入

# MockLLM: 看到 prompt 里有 diff 文本就给出评审, 看到 问好 就回问候
llm = MockLLM(rules=[
    ('diff', {'type': 'text', 'text': '评审意见: 变量命名可更清晰。'}),
    ('问好', {'type': 'text', 'text': '你好呀!'}),
], default={'type': 'text', 'text': '我不确定。'})

r1 = run_slash('/review app.py', router, llm, fake_runner, fake_reader)
r2 = run_slash('/greet Bob', router, llm, fake_runner, fake_reader)
r3 = run_slash('/nope', router, llm, fake_runner, fake_reader)
print('review ->', r1)
print('greet  ->', r2)
print('未知   ->', r3)
assert r1['text'].startswith('评审意见')     # diff 被注入 -> 命中评审规则
assert r2['text'] == '你好呀!'
assert r3['text'].startswith('[命令错误]')   # 未知命令优雅降级
print('✅ slash 命令端到端: 命令->展开 prompt->模型回应; 错误优雅降级')

---
## ✏️ 练习 1：支持命名空间命令（`/git:commit`）

真实命令可命名空间化：`/git:commit`、`/project:deploy`。扩展解析，把命令名里的 `:` 拆成 `namespace` + `command`（无 `:` 时 namespace 为 `None`）。

实现 `parse_namespaced(line)`：在 `parse_command` 基础上，返回 `{'namespace', 'command', 'args'}`（非命令仍返回 `None`）。

In [ ]:
def parse_namespaced(line):
    # TODO: 先复用 parse_command 拿到 {'name','args'}(或 None)
    #       再把 name 按第一个 ':' 拆成 namespace + command
    #       'git:commit' -> namespace='git', command='commit'
    #       'review'     -> namespace=None,  command='review'
    #       返回 {'namespace':..., 'command':..., 'args':...} 或 None
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
a = parse_namespaced('/git:commit "feat: x"')
assert a == {'namespace': 'git', 'command': 'commit', 'args': ['feat: x']}
b = parse_namespaced('/review app.py')
assert b == {'namespace': None, 'command': 'review', 'args': ['app.py']}
assert parse_namespaced('不是命令') is None
print('✅ 练习 1 通过: 命名空间命令 /ns:cmd 正确拆分, 无命名空间时为 None')

## ✏️ 练习 2：默认值占位符 `${1:-fallback}`

模板作者常想给可选参数一个**默认值**：`${1:-main}` 表示『若 $1 缺失或为空，用 main』。

实现 `bind_with_defaults(template, args)`：在普通绑定前，先处理所有 `${N:-默认值}`——存在第 N 个参数且非空就用它，否则用默认值。（提示：先用正则处理带默认值的，再调 `bind_args` 处理剩余普通占位符。）

In [ ]:
def bind_with_defaults(template, args):
    # TODO: 用正则匹配 ${N:-默认值}, 例如 r'\$\{(\d+):-([^}]*)\}'
    #   - 若 1<=N<=len(args) 且 args[N-1] 非空 -> 用 args[N-1]
    #   - 否则 -> 用默认值
    # 处理完带默认值的占位符后, 调用 bind_args(剩余, args) 处理普通 $1/$ARGUMENTS
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
t = '部署到 ${1:-main} 分支; 普通: $2'
assert bind_with_defaults(t, ['prod', 'now']) == '部署到 prod 分支; 普通: now'
assert bind_with_defaults(t, []) == '部署到 main 分支; 普通: '   # $1 缺 -> 默认 main; $2 缺 -> 空
assert bind_with_defaults('分支 ${1:-main}', ['']) == '分支 main'  # 空参数也用默认
print('✅ 练习 2 通过: ${N:-默认值} 在缺失/空时回退默认, 否则用参数')

## ✏️ 练习 3：注入预算控制（截断过长注入）

`!`shell`` 或 `@file` 可能注入海量内容撑爆上下文。给注入加**预算**：单个注入超过 `max_chars` 就截断并加省略标记。

实现 `inject_budgeted(text, runner, reader, max_chars)`：在 `inject_dynamic` 的基础上，对每个注入结果若超 `max_chars` 就截到 `max_chars` 并追加 `...[截断]`。

In [ ]:
def inject_budgeted(text, runner, reader, max_chars=20):
    # TODO: 仿 inject_dynamic, 但每次 runner/reader 的返回若 len>max_chars,
    #       截断为 result[:max_chars] + '...[截断]'
    #   提示: 写一个 clamp(s) 助手, 在两处 re.sub 的 lambda 里套上它
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
def big_runner(cmd):
    return 'x' * 100                      # 故意超长
out = inject_budgeted('日志: !`dump`', big_runner, fake_reader, max_chars=20)
assert out == '日志: ' + 'x' * 20 + '...[截断]'
# 不超长的不截断
out2 = inject_budgeted('短: !`date`', lambda c: 'ok', fake_reader, max_chars=20)
assert out2 == '短: ok'
print('✅ 练习 3 通过: 注入预算控制 —— 超长截断+标记, 不超长原样(与模块01 token预算同构)')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def parse_namespaced(line):
    parsed = parse_command(line)
    if parsed is None:
        return None
    name = parsed['name']
    if ':' in name:
        ns, cmd = name.split(':', 1)
    else:
        ns, cmd = None, name
    return {'namespace': ns, 'command': cmd, 'args': parsed['args']}

In [ ]:
# 练习 2 参考答案
def bind_with_defaults(template, args):
    def repl(m):
        i, default = int(m.group(1)), m.group(2)
        if 1 <= i <= len(args) and args[i - 1] != '':
            return args[i - 1]
        return default
    out = re.sub(r'\$\{(\d+):-([^}]*)\}', repl, template)
    return bind_args(out, args)

In [ ]:
# 练习 3 参考答案
def inject_budgeted(text, runner, reader, max_chars=20):
    def clamp(s):
        return s if len(s) <= max_chars else s[:max_chars] + '...[截断]'
    text = re.sub(r'!`([^`]+)`', lambda m: clamp(runner(m.group(1))), text)
    text = re.sub(r'@(\S+)', lambda m: clamp(reader(m.group(1))), text)
    return text

---
## 🧪 真实数据胶囊：真实 Claude Code 命令文件的形状

下面是一个**贴近真实**的 Claude Code slash 命令文件（`.claude/commands/review.md`）：`---` frontmatter（`description`/`argument-hint`）+ 带 `$ARGUMENTS`、`` !`git diff` ``、`@file` 的模板。我们用上面从零写的解析器/绑定器/注入器去解析并展开它，体会真实命令与本课的逐行对应。

> 形状对照：真实命令文件的 frontmatter 用与模块 01 的 SKILL.md **同一套** `---` 语法——你写的 frontmatter 解析器可以复用。

In [ ]:
# 真实风格的命令文件内容(贴近 Claude Code 文档样例)
REAL_CMD_FILE = '''---
description: 对指定文件做代码评审
argument-hint: <file>
---
请评审文件 $1。
当前 diff:
!`git diff $1`
对照评审清单:
@docs/checklist.md'''

def split_frontmatter(text):
    '''极简 frontmatter 拆分(与模块01同源): 返回 (meta_dict, body)。'''
    meta = {}
    if text.startswith('---'):
        _, fm, body = text.split('---', 2)
        for line in fm.strip().splitlines():
            if ':' in line:
                k, v = line.split(':', 1)
                meta[k.strip()] = v.strip()
        return meta, body.lstrip('\n')
    return meta, text

meta, body = split_frontmatter(REAL_CMD_FILE)
print('frontmatter:', meta)
# 把命令文件注册进路由, 用真实形状跑一遍
router.register('review2', body, meta)
result = router.dispatch('/review2 app.py', fake_runner, fake_reader)
print('展开后的 prompt:\n', result['prompt'])
assert meta['description'] == '对指定文件做代码评审'
assert meta['argument-hint'] == '<file>'
assert result['ok'] is True
assert 'app.py' in result['prompt']                 # $1 绑定
assert '- old\n+ new' in result['prompt']           # !`git diff app.py` 注入
assert '有测试吗' in result['prompt']                 # @checklist 注入
print('✅ 本课解析器/绑定器/注入器直接适用于真实 Claude Code 命令文件')

**🧪 胶囊练习**：实现 `command_summary(cmd_files)`：给定一组命令文件内容，返回 `{命令名(从假设的文件名推断这里用 description 首词省略, 简单用 description): description}` —— 简化为返回 `{description: argument-hint}` 的字典，用于生成 `/help` 列表。（真实 `/help` 就是这样从各命令的 frontmatter 汇总出来的。）

In [ ]:
def command_summary(cmd_files):
    # TODO: 对每个命令文件内容, split_frontmatter 取 meta,
    #       返回 {meta['description']: meta.get('argument-hint','')}
    raise NotImplementedError

In [ ]:
# 自测
summary = command_summary([REAL_CMD_FILE])
assert summary == {'对指定文件做代码评审': '<file>'}
print('命令摘要(供 /help):', summary)
print('✅ 胶囊练习通过')

In [ ]:
# 📖 胶囊参考答案
def command_summary(cmd_files):
    out = {}
    for text in cmd_files:
        meta, _ = split_frontmatter(text)
        out[meta.get('description', '')] = meta.get('argument-hint', '')
    return out

---
## 🔧 旁注：把展开后的 prompt 交给真实 Claude（无 key 自动回退）

slash 命令的产物就是一段 prompt。换成真实 Claude 只是把『喂给模型』那一步替换掉——**有 `ANTHROPIC_API_KEY` 走真实 `messages.create`，没有就回退 MockLLM**，逻辑一行不改。

In [ ]:
def make_llm(rules=None, default=None, model='claude-sonnet-4-6'):
    '''统一 LLM 工厂: 有 key 用真实 Claude, 否则 MockLLM。全课共用此范式。'''
    if os.environ.get('ANTHROPIC_API_KEY'):
        try:
            import anthropic
            client = anthropic.Anthropic()
            def real_llm(prompt):
                msgs = prompt if isinstance(prompt, list) else [{'role': 'user', 'content': prompt}]
                resp = client.messages.create(model=model, max_tokens=1024, messages=msgs)
                for block in resp.content:
                    if block.type == 'tool_use':
                        return {'type': 'tool_use', 'name': block.name, 'input': block.input}
                return {'type': 'text', 'text': ''.join(b.text for b in resp.content if b.type == 'text')}
            print(f'[make_llm] 真实 Claude: {model}')
            return real_llm
        except Exception as e:
            print(f'[make_llm] 真实 Claude 不可用({type(e).__name__}), 回退 MockLLM')
    print('[make_llm] 无 key, 用 MockLLM')
    return MockLLM(rules or [], default=default)

# 本课环境无 key -> 拿到 MockLLM, 但形状与真实一致
llm2 = make_llm(rules=[('diff', {'type': 'text', 'text': '评审意见: 命名可更清晰。'})])
out = run_slash('/review app.py', router, llm2, fake_runner, fake_reader)
assert out['text'].startswith('评审意见')
print('✅ make_llm: 无 key 回退 MockLLM, 有 key 走真实 Claude —— slash 命令逻辑零改动')
print()
print('# 真实 Claude 调用形状(伪代码, 需 key):')
print('''import anthropic
client = anthropic.Anthropic()                       # 读 ANTHROPIC_API_KEY
res = router.dispatch("/review app.py", runner, reader)
resp = client.messages.create(
    model="claude-sonnet-4-6", max_tokens=1024,
    messages=[{"role": "user", "content": res["prompt"]}],  # 展开后的 prompt 原样可用!
)''')

### 小结
- slash 命令 = 用户**显式触发**的能力入口（对照 skill 的模型自动触发）：把常重复工作流固化成带占位符+动态注入的模板。
- **解析**用 `shlex`（懂引号）；**绑定** `$ARGUMENTS`/`$1`（越界填空、多位数 `$11` 正确）；**动态注入** `` !`shell` ``/`@file`（执行器可插拔、安全可测）；**路由**靠注册表、错误返回 `ok=False` 绝不崩。
- **动态注入是最大安全面**：`` !`shell` `` 是任意命令执行，必须白名单/沙箱/确认。
- 产物是**一段展开后的 prompt**，与普通输入无异，`make_llm`（无 key 回退 MockLLM）原样适用。
- 同一套 **frontmatter 解析**与 **注册表 + 分发** 模式贯穿 skill(01)→命令(02)→工具/MCP(03)→插件(04)→系统(05)。

下一站：**模块 03 · 从零造 MCP Server** —— 把『能力从哪来』标准化成一套开放协议(JSON-RPC over stdio)。